# Phase 3 — Is the efficiency score real?

*"I didn't want to invent a number, so I logged every fill-up and checked whether my score actually predicted MPG."*

This notebook regresses the Driveby efficiency score against **ground-truth fuel economy**:

* **Tank-to-tank** (fuel log): each full tank = one point. Expect n ≈ 9 by October.
* **Per-trip OBD-II** (Decision 4): each drive = one point (hundreds).

Target = MPG ÷ EPA combined, which controls for the car. **Always report n next to r².**
"n=9, r²=0.71 — suggestive, and the obvious next step is more data" beats a bare r²=0.94.

**Inputs**
1. Export drives from the app (Drive → *Export raw data for the Python processor*), unzip into `data/`.
2. `python -m processor.run data/ --all` → writes `analysis.json` into each drive folder.
3. Fuel log CSV: `date, odometer, gallons, price_total, car_id[, filled_to_full]` (Fuel tab → share icon exports this format).

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from processor import contract as C, validate as V
from processor.scoring import ScoringModel

DATA = ROOT / "data"
FUEL_CSV = DATA / "fuel_log.csv"
EPA_COMBINED = None      # e.g. 32 for your car — set this! (Settings → Vehicles in the app shows it)
OUT = ROOT / "notebooks" / "out"; OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
drives = V.load_processed_drives(DATA)
print(f"{len(drives)} processed drives, {drives.miles.sum():.0f} miles" if len(drives) else "No processed drives yet — see the instructions above.")
drives.head()

## 1 · Normalise components across the corpus (p5/p95 — fixed constants don't transfer between cars)

In [ ]:
base = ScoringModel().with_corpus(drives[list(C.COMPONENT_KEYS)].to_dict("records")) if len(drives) else ScoringModel()
pd.DataFrame(base.normalization, index=["p5", "p95"])

## 2 · Tank-to-tank intervals

In [ ]:
fuel = V.load_fuel_log(FUEL_CSV) if FUEL_CSV.exists() else None
tanks = V.tank_intervals(fuel, drives) if fuel is not None else pd.DataFrame()
tanks

Coverage = recorded miles ÷ odometer miles. Tanks where you recorded less than half the driving
say little about your score, so they're excluded from the fit (but shown in grey).

In [ ]:
MIN_COVERAGE = 0.5
use = tanks[(tanks.coverage >= MIN_COVERAGE)].dropna(subset=list(C.COMPONENT_KEYS)) if len(tanks) else tanks
y = (use.mpg / EPA_COMBINED if EPA_COMBINED else use.mpg) if len(use) else pd.Series(dtype=float)
scores = [base.efficiency_score(r) for r in use[list(C.COMPONENT_KEYS)].to_dict("records")] if len(use) else []
if len(scores) >= 3:
    lr = stats.linregress(scores, y)
    print(V.honest_summary(len(scores), lr.rvalue**2), f"slope={lr.slope:.4f}")
else:
    print(f"n={len(scores)} — keep logging fill-ups (fill to the first click!).")

## 3 · Per-trip OBD-II (preferred when available)

In [ ]:
obd = drives.dropna(subset=["obd_fuel_gal"]) if "obd_fuel_gal" in drives else drives.iloc[0:0]
obd = obd[obd.obd_fuel_gal > 0.01] if len(obd) else obd
if len(obd):
    obd = obd.assign(mpg=obd.miles / obd.obd_fuel_gal)
    obd = obd[(obd.mpg > 3) & (obd.mpg < 150)]
    y_obd = obd.mpg / EPA_COMBINED if EPA_COMBINED else obd.mpg
    s_obd = [base.efficiency_score(r) for r in obd[list(C.COMPONENT_KEYS)].to_dict("records")]
    lr_obd = stats.linregress(s_obd, y_obd)
    print(V.honest_summary(len(s_obd), lr_obd.rvalue**2))
else:
    print("No OBD fuel data. Decision 4: a $15 ELM327 BLE dongle turns n≈9 into n≈200.")

## 4 · Fit the weights (Step 3.3)

Regress the target on the normalised components, negate and rescale the coefficients into weights.
Ridge regularisation keeps a small n from producing wild weights. **In-sample r² flatters you** —
with more data, hold out the last few tanks and report that r² instead.

In [ ]:
rows, target = (obd, y_obd.to_numpy()) if len(obd) >= 10 else (use, y.to_numpy() if len(use) else np.array([]))
fitted = V.fit_weights(rows, target, base) if len(rows) >= 6 else None
model = fitted or base
print("weights:", {k: round(v, 3) for k, v in model.weights.items()}, "| source:", model.source)

## 5 · The chart for the video (scatter, r² and n annotated) + residuals

In [ ]:
if len(rows):
    sc = [model.efficiency_score(r) for r in rows[list(C.COMPONENT_KEYS)].to_dict("records")]
    lr = V.plot(sc, target, "Actual MPG ÷ EPA combined" if EPA_COMBINED else "Actual MPG",
                "Driveby efficiency score vs real fuel economy", OUT / "validation.png", OUT / "residuals.png")
    model.save(OUT / "scoring_model.json")
    from IPython.display import Image, display
    display(Image(str(OUT / "validation.png"))); display(Image(str(OUT / "residuals.png")))
    print(V.honest_summary(len(sc), None if lr is None else lr.rvalue**2))
    print("Import", OUT / "scoring_model.json", "in the app: Settings → Scoring model → Import.")

## 6 · Say it honestly (paste into the submission / video)

> "The efficiency score is validated against real fuel consumption: n = __, r² = __ — suggestive, and
> the obvious next step is more data. The safety score is not validated — it's built from established
> telematics thresholds, and validating it would require crash data I don't have access to."

If the residual plot shows structure (a curve, a fan), the model is missing something — look at
high-speed share and idle first.